In [70]:
import torch
import cv2
import numpy as np

import os
from extraction.images import model_loader
from extraction.video_processing import extract_video_features_compressed_ms

import time

In [71]:
ucf_path = 'data/images/violence'
violence_dir = ['Violent']
non_violence_dir = ['Normal']

In [72]:
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

In [73]:
# 1. Device Guard Setup
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"⏳ Loading transformer weights into active hardware storage space...")

# 2. Initialize weights ONCE at the top level of the cell
global_model, global_processor, active_code = model_loader(model_code='clip', device=device)
print(f"✅ Transformer successfully cached on: {device.upper()}\n")

⏳ Loading transformer weights into active hardware storage space...
✅ Transformer successfully cached on: MPS



In [74]:
violent_data = []

# ─── PROCESS VIOLENT DATASET TRACK ───
for video_dir in violence_dir:
    dir_path = os.path.join(ucf_path, video_dir)
    if not os.path.isdir(dir_path): continue
    
    for video_file in os.listdir(dir_path):
        # Skip system garbage metadata files like .DS_Store
        if video_file.startswith('.'): continue 

        video_path = os.path.join(dir_path, video_file)
        print(f"Processing: {video_path}")
        
        start_timer = time.time()
        
        # 🔥 FIXED: Passing global_model and global_processor variables smoothly instead of string codes
        vector = extract_video_features_compressed_ms(
            video_path=video_path,
            model=global_model,
            processor=global_processor,
            model_code=active_code,
            target_frames=28
        )
        
        violent_data.append(vector)
        execution_speed = time.time() - start_timer

        print("📊 APPLE SILICON PERFORMANCE AUDIT:")
        print(f"🎬 Video Evaluated: {video_path.split('/')[-1]}")

Processing: data/images/violence/Violent/Abuse019_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Abuse019_x264.mp4
Processing: data/images/violence/Violent/Abuse018_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Abuse018_x264.mp4
Processing: data/images/violence/Violent/Explosion003_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Explosion003_x264.mp4
Processing: data/images/violence/Violent/Fighting009_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Fighting009_x264.mp4
Processing: data/images/violence/Violent/Fighting036_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Fighting036_x264.mp4
Processing: data/images/violence/Violent/Fighting037_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Fighting037_x264.mp4
Processing: data/images/violence/Violent/Fighting003_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Fighting003_x264.mp4
Processing: data/images/violence/Violent/Arrest015_

In [75]:
non_violent_data = []

# ─── PROCESS VIOLENT DATASET TRACK ───
for video_dir in non_violence_dir:
    dir_path = os.path.join(ucf_path, video_dir)
    if not os.path.isdir(dir_path): continue
    
    for video_file in os.listdir(dir_path):
        # Skip system garbage metadata files like .DS_Store
        if video_file.startswith('.'): continue 

        video_path = os.path.join(dir_path, video_file)
        print(f"Processing: {video_path}")
        
        start_timer = time.time()
        
        # 🔥 FIXED: Passing global_model and global_processor variables smoothly instead of string codes
        vector = extract_video_features_compressed_ms(
            video_path=video_path,
            model=global_model,
            processor=global_processor,
            model_code=active_code,
            target_frames=28
        )
        
        non_violent_data.append(vector)
        execution_speed = time.time() - start_timer

        print("📊 APPLE SILICON PERFORMANCE AUDIT:")
        print(f"🎬 Video Evaluated: {video_path.split('/')[-1]}")

Processing: data/images/violence/Normal/Normal_Videos_606_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Normal_Videos_606_x264.mp4
Processing: data/images/violence/Normal/Normal_Videos_310_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Normal_Videos_310_x264.mp4
Processing: data/images/violence/Normal/Normal_Videos_781_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Normal_Videos_781_x264.mp4
Processing: data/images/violence/Normal/Normal_Videos_129_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Normal_Videos_129_x264.mp4
Processing: data/images/violence/Normal/Normal_Videos_189_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Normal_Videos_189_x264.mp4
Processing: data/images/violence/Normal/Normal_Videos_050_x264.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Video Evaluated: Normal_Videos_050_x264.mp4
Processing: data/images/violence/Normal/Normal_Videos_189_x264 copy.mp4
📊 APPLE SILICON PERFORMANCE AUDIT:
🎬 Vid

In [76]:
from sklearn.model_selection import train_test_split
from training.kfold_train import stratified_kfold_train_val

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.metrics import confusion_matrix, classification_report

In [77]:
violent_labels = np.ones(len(violent_data))
non_violent_labels = np.zeros(len(non_violent_data[:9]))

violent_data = np.array(violent_data)
non_violent_data = np.array(non_violent_data[:9])

In [78]:
x = np.concat([violent_data, non_violent_data])
y = np.concat([violent_labels, non_violent_labels])

In [79]:
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size=0.2)

In [80]:
log_reg = LogisticRegression(
    penalty='l2',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='liblinear',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)

stratified_kfold_train_val(5,
                           0.35,
                           log_reg,
                           x_train,
                           y_train)

Starting 5-Fold Stratified Cross-Validation...

sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 1.0000 (When flagged positive, accuracy is 100.00%)
Custom Recall Score:    1.0000 (Captured 100.00% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.8750 (When flagged positive, accuracy is 87.50%)
Custom Recall Score:    1.0000 (Captured 100.00% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.8750 (When flagged positive, accuracy is 87.50%)
Custom Recall Score:    1.0000 (Captured 100.00% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 1.0000 (When flagged positive, accuracy is 100.00%)
Custom Recall Score:    1.0000 (Captured 100.00% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.7500 (When flagged positive, accuracy is 75.00%)
Cu

In [82]:
log_reg = LogisticRegression(
    penalty='l2',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='liblinear',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)
threshold = 0.35

log_reg.fit(x_train, y_train)
test_probabilities = log_reg.predict_proba(x_test)[:, 1]
predictions = (test_probabilities >= threshold).astype(int)

print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))

              precision    recall  f1-score   support

         0.0       1.00      0.67      0.80         3
         1.0       0.89      1.00      0.94         8

    accuracy                           0.91        11
   macro avg       0.94      0.83      0.87        11
weighted avg       0.92      0.91      0.90        11

[[2 1]
 [0 8]]


In [68]:
import joblib

In [69]:
joblib.dump(log_reg, open("model/violence.jobllib", 'wb'))